# Weekly Challenge 04 – Flight Planner ✈️
*Time-series forecasting · difficulty ★★★★ · ~1–2 evenings*

It is December 1958 and an airline has to plan aircraft and crews for the next two years. You have the
**monthly number of international passengers (thousands) from 1949 to 1958** and must forecast **every month of 1959 and 1960**.

The planners currently use the **seasonal naive** rule – "same month last year" – which ignores the growth of air travel.
Your job: **reach a MASE of 0.80 or lower** on the 24-month holdout. That is less than a third of the baseline error.

| | |
|---|---|
| **Data** | Box & Jenkins *airline passengers*, 144 months, [CSV from jbrownlee/Datasets](https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv) (≈ 2 KB download) |
| **Split** | train = 1949-01 … 1958-12 (120 months), test = 1959-01 … 1960-12 (24 months) – fixed, chronological |
| **Metric** | MASE: test MAE divided by the in-sample MAE of the seasonal naive forecast (m = 12) on the training data. Lower is better; < 1 beats "same month last year" *on the training data*. |
| **Baseline** | seasonal naive → **2.494** |
| **Target** | **≤ 0.80** |

Theory: [[Time Series Forecasting]] · [[Time Series Validation and Features]] · [[Classical Forecasting Models]] · [[Linear Regression]]

**Interface** – in this challenge you don't write an estimator but a function `forecast(train, horizon)` that gets a 1-D NumPy array of past values
and returns an array of `horizon` future values. The evaluate cell calls it **once** with all 120 training months.

**How it works**
1. Read the story, run the setup and the baseline.
2. Build your model in the ✏️ *Your turn* cell (it starts as a copy of the baseline).
3. Run the 🏁 *Evaluate* cell – it prints ✅ if you beat the target, ❌ otherwise.
4. Stuck? Open the 💡 hints one at a time. Done (or truly stuck)? Open the 🏆 solution walkthrough.
5. Log your best score in [[Weekly Challenges]].

**Rules** – make every modelling decision with the training data only (a validation window cut from the *end* of the training series).
The test set is touched exactly once, by the evaluate cell. Peeking at it to tune is cheating yourself.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")

def mase(y_true, y_pred, y_train, m=12):
    """Mean absolute scaled error (Hyndman & Koehler, 2006) with a seasonal-naive scale."""
    scale = np.mean(np.abs(y_train[m:] - y_train[:-m]))
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / scale)

def report(name, score, baseline, target, higher_is_better=True):
    beat = score >= target if higher_is_better else score <= target
    arrow = "≥" if higher_is_better else "≤"
    print(f"{name}: {score:.4f}   (baseline {baseline:.4f}, target {arrow} {target})")
    print(("✅ Target beaten – log it in Weekly Challenges!" if beat
           else "❌ Not yet – keep going (or open a hint)."))

## 1 · Load the data (fixed split)

In [ ]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
series = pd.read_csv(URL, parse_dates=["Month"], index_col="Month")["Passengers"].astype(float)
HORIZON = 24
y_train, y_test = series.values[:-HORIZON], series.values[-HORIZON:]
dates_train, dates_test = series.index[:-HORIZON], series.index[-HORIZON:]
print(len(y_train), "train months:", dates_train[0].date(), "→", dates_train[-1].date())
print(len(y_test), "test months: ", dates_test[0].date(), "→", dates_test[-1].date())

## 2 · Have a look
Two questions: is the trend a straight line? And does the seasonal swing stay the same size, or grow with the level?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(dates_train, y_train); ax[0].set_title("passengers (train)")
ax[1].plot(dates_train, np.log(y_train)); ax[1].set_title("log(passengers)")
plt.tight_layout(); plt.show()

## 3 · Baseline – seasonal naive

In [ ]:
def seasonal_naive(train, horizon, m=12):
    return np.array([train[len(train) - m + (h % m)] for h in range(horizon)])

BASELINE = mase(y_test, seasonal_naive(y_train, HORIZON), y_train)
TARGET = 0.80
print(f"baseline test MASE: {BASELINE:.3f}")

## 4 · ✏️ Your turn
Write `forecast(train, horizon)`. To compare ideas **without touching the test set**, cut the last 24 months off `y_train`
as a validation window (the helper below does exactly that).

In [ ]:
def validate(f):
    """MASE of forecaster f on the last 24 months of the TRAINING data."""
    fit_part, val_part = y_train[:-HORIZON], y_train[-HORIZON:]
    return mase(val_part, f(fit_part, HORIZON), fit_part)

print(f"seasonal naive – validation MASE: {validate(seasonal_naive):.3f}")

In [ ]:
# ✏️ TODO: replace this with your own forecaster
def forecast(train, horizon):
    return seasonal_naive(train, horizon)

print(f"validation MASE: {validate(forecast):.3f}")

## 5 · 🏁 Evaluate

In [ ]:
pred = np.asarray(forecast(y_train, HORIZON), dtype=float)
assert pred.shape == (HORIZON,), f"forecast must return {HORIZON} values, got shape {pred.shape}"
score = mase(y_test, pred, y_train)
report("test MASE", score, BASELINE, TARGET, higher_is_better=False)

plt.figure(figsize=(9, 3.5))
plt.plot(dates_train[-48:], y_train[-48:], label="train")
plt.plot(dates_test, y_test, "k", label="actual")
plt.plot(dates_test, seasonal_naive(y_train, HORIZON), "--", label="seasonal naive")
plt.plot(dates_test, pred, label="your forecast"); plt.legend(); plt.show()

<details><summary>💡 Hint 1</summary>

Seasonal naive repeats last year, so it is always one year of growth behind. You need a **trend** plus a **seasonal pattern**.
A linear regression on "time index + 11 month dummies" is a good first forecaster.
</details>

<details><summary>💡 Hint 2</summary>

The seasonal swing grows with the level (see the plot) – that's *multiplicative* seasonality. Model `log(y)` and take `exp` of the forecast.
</details>

<details><summary>💡 Hint 3</summary>

Is the growth constant over all ten years? Compare a few trend shapes (quadratic trend, or fitting only the most recent years) with `validate` and pick the best one *on validation*.
</details>

<details>
<summary>🏆 <b>Solution walkthrough</b> (click to open – try first!)</summary>

**Why the baseline is weak.** Air travel grew ≈ 13 % a year in the 1950s (but only 3 % in 1958). "Same month last year" is therefore always one
year behind – and for the second forecast year, two years behind. Its errors grow with the horizon.

**Step 1 – trend + season regression.** Regress passengers on a time index plus 11 month dummies. On the raw scale
this assumes the seasonal swing is a fixed number of passengers, which the plot contradicts; on the log scale the swing
is a fixed *percentage*. Surprise: on validation both score ≈ 1.0, and on the test set both *miss* the target (≈ 1.21 raw, ≈ 1.50 log).
Why? A straight line fitted to all ten years over-estimates the recent growth rate, because growth was **slowing down**.

**Step 2 – let the trend bend, and choose on validation.** Candidates: quadratic trend in log space, or a linear trend
fitted only on the most recent 4 years. The code below scores every candidate on the last 24 *training* months and
picks the winner automatically:

| candidate | validation MASE | test MASE (for curiosity – not used to choose!) |
|---|---|---|
| linear, last 4 years | **0.749** ← picked | **0.692** ✅ |
| log + quadratic trend | 0.760 | 0.552 |
| log + linear, last 4 years | 0.776 | 0.346 |
| linear trend | 0.957 | 1.212 |
| log + linear trend | 0.999 | 1.504 |

The validation-selected model beats the target with **test MASE ≈ 0.69**. Note how noisy selection on a 24-month
window is: the three bending models are within 0.03 on validation but differ by a factor two on test. A rolling-origin
evaluation (several validation windows) would give a more reliable ranking – see [[Time Series Validation and Features]].

**Going further.** Holt–Winters with multiplicative seasonality or a SARIMA "airline model" (0,1,1)(0,1,1)₁₂ on log data
(the classic Box–Jenkins treatment of exactly this series) are the textbook answers – try them with `statsmodels`.

**Lesson.** In forecasting, the trend you extrapolate matters more than anything else – and model selection must use a
window *after* the data you fit on, never a random split.

```python
from sklearn.linear_model import LinearRegression

def trend_season_model(train, horizon, log=True, degree=1, last=None):
    """Regress (log) passengers on a polynomial trend + month dummies; extrapolate `horizon` months."""
    n = len(train)
    t = np.arange(n + horizon)
    X = np.column_stack([(t / 100) ** d for d in range(1, degree + 1)] + [(t % 12 == k) for k in range(1, 12)])
    start = 0 if last is None else n - last                  # optionally fit on the last `last` months only
    z = np.log(train) if log else train
    reg = LinearRegression().fit(X[start:n], z[start:])
    pred = reg.predict(X[n:])
    return np.exp(pred) if log else pred

candidates = {
    "linear trend":            lambda tr, h: trend_season_model(tr, h, log=False),
    "log + linear trend":      lambda tr, h: trend_season_model(tr, h),
    "log + quadratic trend":   lambda tr, h: trend_season_model(tr, h, degree=2),
    "linear, last 4 years":    lambda tr, h: trend_season_model(tr, h, log=False, last=48),
    "log + linear, last 4 yrs": lambda tr, h: trend_season_model(tr, h, last=48),
}

# honest model selection: hold out the LAST 24 months of the training data as a validation window
fit_part, val_part = y_train[:-HORIZON], y_train[-HORIZON:]
val_scores = {name: mase(val_part, f(fit_part, HORIZON), fit_part) for name, f in candidates.items()}
for name, s in sorted(val_scores.items(), key=lambda kv: kv[1]):
    print(f"validation MASE {s:.3f}   {name}")

best = min(val_scores, key=val_scores.get)
print("→ picked:", best)
forecast = candidates[best]           # the evaluate cell refits it on all of y_train
```
</details>